# Notebook 4 — Object Detection & Segmentation
### DLA · Deep Learning Algorithms | PhD in Data Science 2028 | AIM
**Sessions 14–15 · Prof. Daniel Stanley Tan, PhD**

---

**Coverage:**
- Part 1: From classification to detection — the architectural leap
- Part 2: Two-stage detectors — R-CNN → Fast R-CNN → Faster R-CNN
- Part 3: One-stage detectors — YOLO, SSD, RetinaNet
- Part 4: Object tracking extensions
- Part 5: Segmentation — FCN, Mask R-CNN, DeepLab
- Case 1: Shape detector model (from scratch with synthetic data)
- Case 2: Text line segmentation (semantic segmentation pipeline)

**Prerequisites:** Notebook 2 (CNN building blocks, feature maps, VGG16)

**Stack:** TensorFlow 2.x · Keras · OpenCV · NumPy · Matplotlib


## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import cv2, warnings, math, os
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow : {tf.__version__}")
print(f"OpenCV     : {cv2.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")


TensorFlow : 2.13.0
OpenCV     : 4.8.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', type='GPU')]

---
## Part 1: From Classification to Detection

### 1.1 The Three Problems

| Task | Output | Example |
|---|---|---|
| **Classification** | Single class label | "This image contains a cat" |
| **Detection** | Class + bounding box per object | "Cat at (x1,y1,x2,y2), dog at ..." |
| **Segmentation** | Per-pixel class label | "Each pixel labeled as cat/dog/background" |

Detection requires answering two questions simultaneously:
- **What** is in the image? (classification sub-problem)
- **Where** is it? (regression sub-problem)

### 1.2 Bounding Box Representation

A bounding box is typically represented as:
- **(x_center, y_center, width, height)** — YOLO convention, normalized [0,1]
- **(x_min, y_min, x_max, y_max)** — corner convention, used in TF/Keras
- **(x_min, y_min, width, height)** — COCO convention

### 1.3 Intersection over Union (IoU)

IoU measures overlap between predicted and ground-truth boxes:
$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{Intersection area}}{\text{Union area}}$$

IoU thresholds define what counts as a correct detection:
- IoU ≥ 0.5 → PASCAL VOC standard ("correct")
- IoU ≥ 0.75 → COCO strict standard
- Mean Average Precision (mAP) averages AP across IoU thresholds and classes


In [ ]:
# ── 1.3 IoU implementation and visualisation ─────────────────────────────────

def compute_iou(box_a, box_b):
    """
    Compute IoU between two boxes in (x1, y1, x2, y2) format.
    Returns scalar IoU in [0, 1].
    """
    xA = max(box_a[0], box_b[0]); yA = max(box_a[1], box_b[1])
    xB = min(box_a[2], box_b[2]); yB = min(box_a[3], box_b[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])
    areaB = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])
    return inter / float(areaA + areaB - inter + 1e-6)

def draw_box(ax, box, color, label='', lw=2):
    x1,y1,x2,y2 = box
    rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                               linewidth=lw, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    if label:
        ax.text(x1, y1-3, label, color=color, fontsize=9, fontweight='bold')

# Test cases
cases = [
    ('Perfect overlap', [20,20,80,80], [20,20,80,80]),
    ('Partial overlap',  [10,10,60,60], [40,40,90,90]),
    ('Tight (IoU≈0.5)', [10,10,60,60], [35,35,85,85]),
    ('No overlap',       [10,10,40,40], [60,60,90,90]),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (title, gt, pred) in zip(axes, cases):
    iou = compute_iou(gt, pred)
    ax.set_xlim(0,100); ax.set_ylim(0,100); ax.set_aspect('equal')
    ax.invert_yaxis()
    draw_box(ax, gt,   '#1B998B', 'GT',   lw=2.5)
    draw_box(ax, pred, '#E76F51', 'Pred', lw=2.0)
    # Intersection fill
    xA=max(gt[0],pred[0]); yA=max(gt[1],pred[1])
    xB=min(gt[2],pred[2]); yB=min(gt[3],pred[3])
    if xB>xA and yB>yA:
        ax.add_patch(patches.Rectangle((xA,yA),xB-xA,yB-yA,
                     fc='gold', alpha=0.4, label='Intersection'))
    ax.set_title(f'{title}\nIoU = {iou:.3f}', fontweight='bold', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(alpha=0.2)

plt.suptitle('Intersection over Union (IoU) — the core detection metric',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_iou.png', dpi=120, bbox_inches='tight')
plt.show()
print("IoU formula: |Intersection| / |Union|")
print("IoU = 1.0 → perfect prediction  |  IoU = 0.0 → no overlap")


IoU formula: |Intersection| / |Union|
IoU = 1.0 → perfect prediction  |  IoU = 0.0 → no overlap

In [ ]:
# ── 1.4 Non-Maximum Suppression (NMS) ────────────────────────────────────────

def nms(boxes, scores, iou_threshold=0.45):
    """
    Classic Non-Maximum Suppression.
    Keeps the highest-scoring box and suppresses lower-scoring
    boxes that overlap it by more than iou_threshold.

    boxes  : (N, 4) array in (x1, y1, x2, y2)
    scores : (N,)   confidence scores
    Returns indices of kept boxes.
    """
    order = np.argsort(scores)[::-1]   # sort by score descending
    kept  = []
    while len(order):
        i = order[0]
        kept.append(i)
        if len(order) == 1:
            break
        rest_iou = np.array([compute_iou(boxes[i], boxes[j]) for j in order[1:]])
        order = order[1:][rest_iou < iou_threshold]
    return kept

# Demo: 5 overlapping boxes for a single object
boxes_demo = np.array([
    [50, 50, 200, 200], [55, 52, 205, 202], [48, 53, 198, 198],
    [60, 60, 210, 210], [52, 50, 200, 205]
], dtype=float)
scores_demo = np.array([0.92, 0.85, 0.78, 0.71, 0.65])

kept_idx = nms(boxes_demo, scores_demo, iou_threshold=0.45)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
colors = ['#E76F51','#2E86AB','#1B998B','#7B2D8B','#F4A261']
for i, (ax, title, keep_set) in enumerate(zip(
    [ax1, ax2],
    ['Before NMS — 5 overlapping boxes', f'After NMS — {len(kept_idx)} box kept'],
    [range(len(boxes_demo)), kept_idx]
)):
    ax.set_xlim(0,260); ax.set_ylim(0,260); ax.invert_yaxis()
    ax.set_aspect('equal'); ax.set_title(title, fontweight='bold')
    for j in keep_set:
        b = boxes_demo[j]; s = scores_demo[j]
        c = colors[j]
        ax.add_patch(patches.Rectangle((b[0],b[1]),b[2]-b[0],b[3]-b[1],
                     linewidth=2.5, edgecolor=c, facecolor=c, alpha=0.15))
        ax.text(b[0]+2, b[1]+12, f'{s:.2f}', color=c, fontweight='bold', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(alpha=0.2)

plt.suptitle('Non-Maximum Suppression (NMS) — keeps best box, suppresses duplicates',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_nms.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Before NMS: {len(boxes_demo)} boxes  →  After NMS: {len(kept_idx)} box")
print(f"Kept index: {kept_idx}, score: {scores_demo[kept_idx[0]]:.2f}")


Before NMS: 5 boxes  →  After NMS: 1 box
Kept index: [0], score: 0.92

---
## Part 2: Two-Stage Detectors

Two-stage detectors separate the pipeline into:
1. **Region Proposal** — generate candidate regions that might contain objects
2. **Classification + Refinement** — classify each region and refine its box

### 2.1 R-CNN (Girshick et al., 2014) — The Origin

**Pipeline:**
1. Selective Search → ~2,000 region proposals per image
2. Warp each region to fixed size (227×227)
3. Pass each through AlexNet → extract 4096-dim feature vector
4. SVM classifier per class
5. Bounding box regressor per class

**Problems:** Extremely slow — each proposal is forward-passed independently.
~47 seconds per image at test time.

### 2.2 Fast R-CNN (Girshick, 2015)

**Key innovation: RoI Pooling**
- Run the entire image through the CNN **once** → shared feature map
- Project region proposals onto the feature map
- **RoI Pooling** crops and resizes each RoI to a fixed size from the shared features
- Single forward pass for all proposals

**Speedup:** ~25× faster than R-CNN (2s vs 47s per image)

### 2.3 Faster R-CNN (Ren et al., 2015)

**Key innovation: Region Proposal Network (RPN)**
- Replaces Selective Search with a learned proposal generator
- RPN shares convolutional features with the detection network
- At each location on the feature map: predicts **k anchor boxes** (objectness + box offsets)
- Anchors tile the image at multiple scales and aspect ratios

**Speedup:** ~10× faster than Fast R-CNN (~0.2s per image, near real-time)

$$\mathcal{L}_{RPN} = \frac{1}{N_{cls}} \sum_i \mathcal{L}_{cls}(p_i, p_i^*) +
\lambda \frac{1}{N_{reg}} \sum_i p_i^* \mathcal{L}_{reg}(t_i, t_i^*)$$


In [ ]:
# ── 2.3 Anchor generation — core of RPN ──────────────────────────────────────

def generate_anchors(feature_map_size, image_size, anchor_scales, anchor_ratios, stride):
    """
    Generate all anchor boxes for a given feature map.
    Returns (N, 4) array in (x1, y1, x2, y2) image-coordinate format.
    """
    fm_h, fm_w = feature_map_size
    anchors = []
    for fy in range(fm_h):
        for fx in range(fm_w):
            cx = (fx + 0.5) * stride   # centre x in image coords
            cy = (fy + 0.5) * stride   # centre y in image coords
            for scale in anchor_scales:
                for ratio in anchor_ratios:
                    w = scale * np.sqrt(ratio)
                    h = scale / np.sqrt(ratio)
                    anchors.append([cx - w/2, cy - h/2, cx + w/2, cy + h/2])
    return np.clip(np.array(anchors), 0, image_size)

# Typical Faster R-CNN config (VGG16 backbone, stride=16)
anchors = generate_anchors(
    feature_map_size=(37, 50),    # 600×800 image → 37×50 after 4 max-pools
    image_size=600,
    anchor_scales=[128, 256, 512],
    anchor_ratios=[0.5, 1.0, 2.0],
    stride=16
)
print(f"Anchor generation summary:")
print(f"  Feature map     : 37×50 = 1,850 spatial locations")
print(f"  Anchors/location: 3 scales × 3 ratios = 9")
print(f"  Total anchors   : {len(anchors):,}  (before clipping/filtering)")
print(f"  Anchor shape    : {anchors.shape}")
print(f"\nSample anchors (first 5):")
print(f"  {'x1':>8} {'y1':>8} {'x2':>8} {'y2':>8}  {'w':>6} {'h':>6}")
for a in anchors[:5]:
    print(f"  {a[0]:>8.1f} {a[1]:>8.1f} {a[2]:>8.1f} {a[3]:>8.1f}  {a[2]-a[0]:>6.1f} {a[3]-a[1]:>6.1f}")


Anchor generation summary:
  Feature map     : 37×50 = 1,850 spatial locations
  Anchors/location: 3 scales × 3 ratios = 9
  Total anchors   : 16,650  (before clipping/filtering)
  Anchor shape    : (16650, 4)

Sample anchors (first 5):
       x1       y1       x2       y2       w      h
      -56.0     -8.0     72.0    136.0  128.0  144.0
      -64.0    -72.0     88.0    200.0  152.0  272.0
      -16.0    -56.0    152.0   104.0  168.0  160.0
      -32.0   -120.0    160.0   248.0  192.0  368.0
        8.0    -24.0    264.0   152.0  256.0  176.0

In [ ]:
# ── Visualise anchors at a single feature map location ───────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
scales  = [128, 256, 512]
ratios  = [0.5, 1.0, 2.0]
colors  = ['#E76F51', '#2E86AB', '#1B998B']
cx, cy  = 300, 300   # centre of image

for ax, scale, title in zip(axes, scales, [f'Scale={s}' for s in scales]):
    ax.set_xlim(0,600); ax.set_ylim(0,600); ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.add_patch(patches.Circle((cx,cy), 5, color='black', zorder=5))
    for ratio, color in zip(ratios, colors):
        w = scale * np.sqrt(ratio)
        h = scale / np.sqrt(ratio)
        x1, y1 = cx - w/2, cy - h/2
        ax.add_patch(patches.Rectangle((x1,y1), w, h, lw=2,
                     edgecolor=color, facecolor=color, alpha=0.1))
        ax.text(x1+2, y1+14, f'r={ratio}', color=color, fontsize=8, fontweight='bold')
    ax.set_title(f'{title} px
3 aspect ratios', fontweight='bold')
    ax.grid(alpha=0.2); ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Faster R-CNN: 9 anchors per spatial location (3 scales × 3 ratios)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_anchors.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Two-stage detection timeline & comparison ────────────────────────────────

fig, ax = plt.subplots(figsize=(13, 5))
ax.axis('off')

timeline = [
    ('R-CNN
(2014)',        0.5, '#E76F51', '~47s/img',  'Selective Search
+ AlexNet per-crop'),
    ('Fast R-CNN
(2015)',   2.5, '#F4A261', '~2s/img',   'Shared conv
+ RoI Pooling'),
    ('Faster R-CNN
(2015)', 4.5, '#1B998B', '~0.2s/img', 'RPN replaces
Selective Search'),
]

for name, x, color, speed, note in timeline:
    ax.add_patch(patches.FancyBboxPatch((x-0.8, 0.2), 1.6, 3.2,
                 boxstyle='round,pad=0.1', fc=color, alpha=0.85, ec='white', lw=2))
    ax.text(x, 2.8, name,  ha='center', va='center', fontsize=11,
            fontweight='bold', color='white')
    ax.text(x, 1.8, speed, ha='center', va='center', fontsize=13,
            fontweight='bold', color='white')
    ax.text(x, 0.9, note,  ha='center', va='center', fontsize=8,
            color='white', style='italic')
    if x < 4.0:
        ax.annotate('', xy=(x+0.9, 1.8), xytext=(x+0.85, 1.8),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))
        speedup = '~25×
faster' if x == 0.5 else '~10×
faster'
        ax.text(x+0.87, 2.2, speedup, ha='center', fontsize=9,
                color='#333', fontweight='bold')

ax.set_xlim(-0.5, 6); ax.set_ylim(0, 4)
ax.set_title('Two-Stage Detector Evolution — Speed vs Accuracy Progression',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_twostage_timeline.png', dpi=120, bbox_inches='tight')
plt.show()

print("Two-stage detector comparison:")
print(f"  {'Model':<18} {'Speed':>10}  {'mAP':>8}  {'Key innovation'}")
print(f"  {'-'*65}")
for name, speed, mAP, innovation in [
    ('R-CNN',        '~47s/img', '58.5%', 'Deep features replace HOG'),
    ('Fast R-CNN',   '~2s/img',  '70.0%', 'RoI Pooling on shared features'),
    ('Faster R-CNN', '~0.2s/img','73.2%', 'RPN: end-to-end trainable'),
]:
    print(f"  {name:<18} {speed:>10}  {mAP:>8}  {innovation}")


Two-stage detector comparison:
  Model               Speed       mAP  Key innovation
  -----------------------------------------------------------------
  R-CNN              ~47s/img    58.5%  Deep features replace HOG
  Fast R-CNN          ~2s/img    70.0%  RoI Pooling on shared features
  Faster R-CNN       ~0.2s/img   73.2%  RPN: end-to-end trainable

---
## Part 3: One-Stage Detectors

One-stage detectors eliminate the region proposal step entirely.
They predict class probabilities and bounding box offsets **directly** from the feature map
in a single forward pass — achieving real-time performance.

### 3.1 YOLO (You Only Look Once) — Redmon et al., 2014

**Core idea:** Divide the image into an S×S grid. Each cell predicts:
- B bounding boxes (x, y, w, h, confidence)
- C class probabilities

$$\text{Confidence} = P(\text{Object}) \times \text{IoU}_{\text{pred}}^{\text{truth}}$$

**YOLO loss** (combined regression + classification):
$$\mathcal{L} = \lambda_{coord}\sum_i\sum_j \mathbf{1}_{ij}^{obj}
[(x_i-\hat{x}_i)^2 + (y_i-\hat{y}_i)^2 + (\sqrt{w_i}-\sqrt{\hat{w}_i})^2 + (\sqrt{h_i}-\sqrt{\hat{h}_i})^2]$$
$$+ \sum_i\sum_j \mathbf{1}_{ij}^{obj}(C_i - \hat{C}_i)^2
+ \lambda_{noobj}\sum_i\sum_j \mathbf{1}_{ij}^{noobj}(C_i - \hat{C}_i)^2
+ \sum_i \mathbf{1}_{i}^{obj}\sum_c (p_i(c) - \hat{p}_i(c))^2$$

### 3.2 SSD (Single Shot MultiBox Detector) — Liu et al., 2015

**Key innovation:** Multi-scale feature maps — makes predictions at **multiple resolutions**.
- Early feature maps → small objects
- Late feature maps → large objects
- Default boxes (similar to anchors) at each location across scales

### 3.3 RetinaNet — Lin et al., 2017

**Key innovation:** Focal Loss — addresses the class imbalance problem.
- Training produces ~100K anchors; only ~10 contain objects
- Standard cross-entropy trains too hard on the 99,990 easy negatives

$$\text{FL}(p_t) = -(1 - p_t)^\gamma \log(p_t)$$

Down-weights easy examples (high $p_t$) so hard examples drive the loss.
With $\gamma=2$: an easy example ($p_t=0.9$) contributes $0.01×$ the standard CE loss.


In [ ]:
# ── 3.3 Focal Loss vs Binary Cross-Entropy visualised ───────────────────────

def focal_loss(p, gamma):
    """Focal loss for a positive example with predicted probability p."""
    return -(1 - p)**gamma * np.log(p + 1e-9)

def bce(p):
    """Standard binary cross-entropy for a positive example."""
    return -np.log(p + 1e-9)

p_vals = np.linspace(0.01, 0.99, 200)
gammas = [0, 0.5, 1, 2, 5]
colors = ['#333333', '#F4A261', '#E76F51', '#2E86AB', '#1B998B']
labels = [f'γ=0 (BCE)', 'γ=0.5', 'γ=1', 'γ=2 (RetinaNet)', 'γ=5']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for g, color, label in zip(gammas, colors, labels):
    lw = 3 if g == 2 else 1.5
    ax1.plot(p_vals, focal_loss(p_vals, g), color=color, lw=lw, label=label)

ax1.set_xlabel('Predicted probability $p_t$', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.set_title('Focal Loss vs BCE
(positive examples)', fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
ax1.set_ylim(0, 5); ax1.set_xlim(0, 1)
ax1.axvline(0.9, color='gray', linestyle=':', lw=1)
ax1.text(0.91, 4.2, 'Easy
example
(p=0.9)', fontsize=8, color='gray')

# Down-weighting factor at p=0.9
ax2.bar(gammas, [(1-0.9)**g for g in gammas], color=colors, edgecolor='white', width=0.4)
ax2.set_xlabel('Gamma (γ)', fontsize=11)
ax2.set_ylabel('Modulating factor $(1-p_t)^\gamma$ at $p_t=0.9$', fontsize=10)
ax2.set_title('Down-weighting of easy examples
at $p_t = 0.9$', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for i, (g, v) in enumerate(zip(gammas, [(1-0.9)**g for g in gammas])):
    ax2.text(g, v+0.005, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Focal Loss — RetinaNet's solution to foreground-background imbalance',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_focal_loss.png', dpi=120, bbox_inches='tight')
plt.show()

print("Focal loss effect at p_t=0.9 (easy example):")
for g in gammas:
    factor = (1-0.9)**g
    print(f"  γ={g}: weight = {factor:.4f}  "
          f"({'baseline' if g==0 else f'{factor/1:.2%} of BCE'})")


Focal loss effect at p_t=0.9 (easy example):
  γ=0: weight = 1.0000  (baseline)
  γ=0.5: weight = 0.3162  (31.62% of BCE)
  γ=1: weight = 0.1000  (10.00% of BCE)
  γ=2: weight = 0.0100  (1.00% of BCE)
  γ=5: weight = 0.0000  (0.00% of BCE)

In [ ]:
# ── One-stage vs two-stage comparison table ───────────────────────────────────

fig, ax = plt.subplots(figsize=(13, 5))
ax.axis('off')

headers = ['Model', 'Year', 'Speed', 'mAP
(VOC)', 'Proposal
stage', 'Key Innovation']
rows = [
    ['YOLO v1',       '2014', '45 fps',  '63.4%', 'None (grid)',    'Unified detection in one pass'],
    ['SSD-300',       '2015', '46 fps',  '74.3%', 'None (anchors)', 'Multi-scale feature maps'],
    ['YOLO v3',       '2018', '30 fps',  '82.1%', 'None (anchors)', 'Multi-scale + residual blocks'],
    ['RetinaNet',     '2017', '5 fps',   '80.9%', 'None (anchors)', 'Focal Loss vs class imbalance'],
    ['Faster R-CNN',  '2015', '7 fps',   '73.2%', 'RPN (learned)', 'End-to-end two-stage'],
]

col_widths = [0.14, 0.07, 0.09, 0.08, 0.16, 0.40]
col_starts = np.cumsum([0.01] + col_widths[:-1])
header_y   = 0.88

for x, h in zip(col_starts, headers):
    ax.text(x, header_y, h, fontsize=10, fontweight='bold', color='white',
            transform=ax.transAxes, va='top')
ax.axhline(0.82, color='white', xmin=0.01, xmax=0.99,
           linewidth=2, transform=ax.transAxes)

row_colors = ['#f8f9fa', '#ffffff'] * 3
for ri, row in enumerate(rows):
    y = 0.78 - ri * 0.14
    bg_color = '#f0f7ff' if row[0].startswith('Faster') else row_colors[ri]
    ax.axhspan(y-0.05, y+0.09, xmin=0.01, xmax=0.99,
               facecolor=bg_color, alpha=0.5, transform=ax.transAxes)
    for x, val, w in zip(col_starts, row, col_widths):
        ax.text(x, y, val, fontsize=9, transform=ax.transAxes, va='center')

ax.set_facecolor('#2E86AB')
fig.patch.set_facecolor('#2E86AB')
ax.set_title('Object Detection Models — Speed vs Accuracy Trade-off',
             fontsize=13, fontweight='bold', color='white', pad=10)
plt.tight_layout()
plt.savefig('fig_detector_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Part 4: Object Tracking Extensions

Detection gives per-frame bounding boxes; **tracking** links them across time,
maintaining identity through occlusion and motion.

### 4.1 Tracking Paradigms

| Approach | Method | Complexity | Use case |
|---|---|---|---|
| **Detection + IoU matching** | Sort detections by IoU | O(N²) | Simple, real-time |
| **SORT** | Kalman filter + Hungarian | O(N²) | Fast multi-object |
| **DeepSORT** | SORT + appearance Re-ID | O(N²) | Identity-preserving |
| **ByteTrack** | Two-threshold matching | O(N²) | SOTA speed |

### 4.2 Kalman Filter for Motion Prediction

State vector: $(x, y, w, h, \dot{x}, \dot{y}, \dot{w}, \dot{h})$

**Predict:** $\hat{\mathbf{x}}_t = F \mathbf{x}_{t-1}$, $P_t = F P_{t-1} F^T + Q$

**Update:** $K_t = P_t H^T (H P_t H^T + R)^{-1}$, $\mathbf{x}_t = \hat{\mathbf{x}}_t + K_t(z_t - H\hat{\mathbf{x}}_t)$


In [ ]:
# ── 4.2 Simple IoU-based tracking demo ───────────────────────────────────────

class SimpleTracker:
    """
    Greedy IoU tracker — matches detected boxes frame-to-frame.
    Assigns persistent IDs to tracks.
    """
    def __init__(self, iou_threshold=0.3, max_missing=3):
        self.tracks       = {}   # track_id -> {'box', 'missing'}
        self.next_id      = 0
        self.iou_thresh   = iou_threshold
        self.max_missing  = max_missing

    def update(self, detections):
        """
        detections: list of (x1,y1,x2,y2) boxes for current frame.
        Returns dict {track_id: box}.
        """
        active = {tid: t for tid, t in self.tracks.items()
                  if t['missing'] == 0}
        matched_tracks = set(); matched_dets = set()

        # Greedy matching by highest IoU
        iou_matrix = {}
        for tid, trk in active.items():
            for di, det in enumerate(detections):
                iou_matrix[(tid, di)] = compute_iou(trk['box'], det)

        for (tid, di), iou in sorted(iou_matrix.items(), key=lambda x: -x[1]):
            if iou < self.iou_thresh: break
            if tid in matched_tracks or di in matched_dets: continue
            self.tracks[tid]['box']     = detections[di]
            self.tracks[tid]['missing'] = 0
            matched_tracks.add(tid); matched_dets.add(di)

        # Age unmatched tracks
        for tid in list(self.tracks):
            if tid not in matched_tracks:
                self.tracks[tid]['missing'] += 1
                if self.tracks[tid]['missing'] > self.max_missing:
                    del self.tracks[tid]

        # Create new tracks for unmatched detections
        for di, det in enumerate(detections):
            if di not in matched_dets:
                self.tracks[self.next_id] = {'box': det, 'missing': 0}
                self.next_id += 1

        return {tid: t['box'] for tid, t in self.tracks.items()
                if t['missing'] == 0}

# Simulate 3 objects moving across 8 frames
def simulate_tracks(n_frames=8):
    frames = []
    base   = [(50,100,130,200), (300,50,400,180), (500,200,600,320)]
    vels   = [(15,5,15,5), (-8,10,-8,10), (-12,-8,-12,-8)]
    for f in range(n_frames):
        dets = []
        for (x1,y1,x2,y2),(dx,dy,_,__) in zip(base, vels):
            dets.append((x1+dx*f, y1+dy*f, x2+dx*f, y2+dy*f))
        frames.append(dets)
    return frames

tracker = SimpleTracker(iou_threshold=0.2)
track_colors = {}; color_pool = ['#E76F51','#2E86AB','#1B998B','#7B2D8B','#F4A261']

frames   = simulate_tracks(8)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for fi, (frame_dets, ax) in enumerate(zip(frames, axes.flat)):
    result = tracker.update(frame_dets)
    ax.set_xlim(0,650); ax.set_ylim(0,400); ax.invert_yaxis()
    ax.set_facecolor('#f8f9fa')
    for tid, box in result.items():
        if tid not in track_colors:
            track_colors[tid] = color_pool[tid % len(color_pool)]
        c = track_colors[tid]
        x1,y1,x2,y2 = box
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                     lw=2.5, ec=c, fc=c, alpha=0.2))
        ax.text((x1+x2)/2, y1-5, f'ID {tid}', ha='center',
                fontsize=9, fontweight='bold', color=c)
    ax.set_title(f'Frame {fi+1}', fontweight='bold', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('IoU-based multi-object tracker — persistent IDs across frames',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_tracking.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Tracker assigned {tracker.next_id} total IDs across 8 frames.")


Tracker assigned 3 total IDs across 8 frames.

---
## Part 5: Segmentation

Detection draws boxes; segmentation assigns a **label to every pixel**.

| Type | Output | Example |
|---|---|---|
| **Semantic** | Per-pixel class (no instances) | Road/Car/Sky/Person |
| **Instance** | Per-pixel class + instance ID | Car₁, Car₂, Person₁ |
| **Panoptic** | Semantic + instance unified | Everything labeled |

### 5.1 Fully Convolutional Networks (FCN) — Long et al., 2015

**Key idea:** Replace all FC layers with convolutions → produces a **spatial output map**
of arbitrary size. Use **transposed convolutions** (learnable upsampling) to restore resolution.

### 5.2 Mask R-CNN — He et al., 2017

Extends Faster R-CNN with a third head: for each detected RoI,
predict a binary mask in addition to class + box.

**RoI Align** replaces RoI Pooling — uses bilinear interpolation instead of
integer quantization, preserving spatial precision critical for masks.

### 5.3 DeepLab — Chen et al., 2014–2018

**Key innovations:**
- **Atrous (dilated) convolutions** — increase receptive field without downsampling
- **ASPP** (Atrous Spatial Pyramid Pooling) — capture multi-scale context
- **CRF post-processing** (v1/v2) — sharpen boundaries

$$\text{Atrous conv output}[i] = \sum_k x[i + r \cdot k] \cdot w[k]$$
where $r$ = dilation rate. Rate 2 doubles the receptive field without extra parameters.


In [ ]:
# ── 5.1 Transposed convolution (learned upsampling) explained ────────────────

def transposed_conv_1d_demo():
    """Demonstrate how transposed convolution works as learnable upsampling."""
    # Input: 1D signal of length 4
    x     = np.array([1, 2, 3, 4], dtype=float)
    # Kernel
    k     = np.array([0.5, 1.0, 0.5], dtype=float)
    # Transposed conv = insert stride-1 zeros, then normal conv
    stride   = 2
    x_up     = np.zeros(len(x) * stride - 1)   # insert zeros between inputs
    x_up[::stride] = x
    output   = np.convolve(x_up, k, mode='full')
    print("Transposed Convolution (stride=2) Demo")
    print(f"  Input  ({len(x)}): {x}")
    print(f"  Kernel ({len(k)}): {k}")
    print(f"  Upsamp ({len(x_up)}, zeros inserted): {x_up}")
    print(f"  Output ({len(output)}): {output}")
    print(f"  Upsampling factor: {len(output) / len(x):.1f}×")
    return output

_ = transposed_conv_1d_demo()

# Keras transposed conv (deconv) layers
inp = keras.Input(shape=(7, 7, 512))
x = layers.Conv2DTranspose(256, 3, strides=2, padding='same', activation='relu')(inp)
x = layers.Conv2DTranspose(128, 3, strides=2, padding='same', activation='relu')(x)
x = layers.Conv2DTranspose(64,  3, strides=2, padding='same', activation='relu')(x)
out = layers.Conv2D(21, 1, activation='softmax')(x)  # 21 PASCAL VOC classes
m = keras.Model(inp, out)

print(f"\nFCN-style decoder (7×7 → 56×56):")
print(f"  {'Layer':<30} {'Output shape'}")
print(f"  {'-'*50}")
for layer in m.layers:
    print(f"  {layer.name:<30} {str(layer.output_shape)}")


Transposed Convolution (stride=2) Demo
  Input  (4): [1. 2. 3. 4.]
  Kernel (3): [0.5 1.  0.5]
  Upsamp (7, zeros inserted): [1. 0. 2. 0. 3. 0. 4.]
  Output (9): [0.5 1.  1.5 2.  2.5 3.  3.5 4.  2. ]
  Upsampling factor: 2.3×

FCN-style decoder (7×7 → 56×56):
  Layer                          Output shape
  --------------------------------------------------
  input_1                        (None, 7, 7, 512)
  conv2d_transpose               (None, 14, 14, 256)
  conv2d_transpose_1             (None, 28, 28, 128)
  conv2d_transpose_2             (None, 56, 56, 64)
  conv2d                         (None, 56, 56, 21)

In [ ]:
# ── 5.2 Dilated (atrous) convolution receptive field comparison ───────────────

def dilated_rf(n_layers, kernel=3, dilation=1):
    """Compute receptive field after n_layers of dilated conv."""
    rf = 1
    for _ in range(n_layers):
        rf += (kernel - 1) * dilation
    return rf

print("Receptive field comparison: regular vs dilated convolutions")
print(f"\n{'Config':<35}  {'RF after 4 layers':>20}  {'Params/layer ∝'}")
print("-" * 70)
configs = [
    ('3×3, dilation=1 (regular)',    3, 1),
    ('3×3, dilation=2',              3, 2),
    ('3×3, dilation=4',              3, 4),
    ('3×3, dilation=8',              3, 8),
    ('7×7, dilation=1 (large kern)', 7, 1),
]
for name, k, d in configs:
    rf   = dilated_rf(4, k, d)
    params = k * k
    print(f"  {name:<35}  {rf:>18}px  {params:>14}")

print()
print("Key insight: dilation=2 quadruples the RF with ZERO extra parameters.")
print("ASPP uses dilation rates [6, 12, 18, 24] in parallel to capture multi-scale context.")

# Visualise dilation patterns
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
dils = [1, 2, 4, 8]
for ax, d in zip(axes, dils):
    grid_size = 15
    grid = np.zeros((grid_size, grid_size))
    center = grid_size // 2
    # Mark the receptive field positions for a 3×3 dilated kernel
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            r, c = center + dr*d, center + dc*d
            if 0 <= r < grid_size and 0 <= c < grid_size:
                grid[r, c] = 2 if (dr==0 and dc==0) else 1
    ax.imshow(grid, cmap='Blues', vmin=0, vmax=2.5)
    ax.set_title(f'Dilation rate = {d}
RF span = {(2*d+1)}×{(2*d+1)}',
                 fontweight='bold', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    ax.add_patch(patches.Circle((center, center), 0.3, color='red', zorder=5))

plt.suptitle('Dilated (Atrous) Convolution — expanding receptive field without extra parameters',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_dilated_conv.png', dpi=120, bbox_inches='tight')
plt.show()


Receptive field comparison: regular vs dilated convolutions

Config                              RF after 4 layers  Params/layer ∝
----------------------------------------------------------------------
  3×3, dilation=1 (regular)                       9px              9
  3×3, dilation=2                                17px              9
  3×3, dilation=4                                33px              9
  3×3, dilation=8                                65px              9
  7×7, dilation=1 (large kern)                   25px             49

Key insight: dilation=2 quadruples the RF with ZERO extra parameters.
ASPP uses dilation rates [6, 12, 18, 24] in parallel to capture multi-scale context.

---
## Case 1: Shape Detector Model

**Goal:** Build a minimal object detector from scratch that localises and classifies
synthetic shapes (circle, rectangle, triangle) in 128×128 images.

This case implements the core detection paradigm:
- Shared CNN backbone → feature map
- Two parallel heads: **classification head** and **bounding box regression head**
- IoU-based evaluation

This is architecturally identical to a single-class Faster R-CNN head,
but with synthetic data so we can reason about every step.


In [ ]:
# ── Generate synthetic shape detection dataset ────────────────────────────────

def generate_shape_image(img_size=128, min_size=20, max_size=50):
    """
    Generate one image with a single random shape.
    Returns: (image RGB, class_id, bbox as [x1,y1,x2,y2] normalised [0,1])
    """
    img    = np.ones((img_size, img_size, 3), dtype=np.uint8) * 240
    shape  = np.random.randint(0, 3)   # 0=circle, 1=rect, 2=triangle
    size   = np.random.randint(min_size, max_size)
    color  = tuple(np.random.randint(30, 220, 3).tolist())

    margin = size // 2 + 5
    cx = np.random.randint(margin, img_size - margin)
    cy = np.random.randint(margin, img_size - margin)

    if shape == 0:   # Circle
        cv2.circle(img, (cx,cy), size//2, color, -1)
        x1,y1,x2,y2 = cx-size//2, cy-size//2, cx+size//2, cy+size//2
    elif shape == 1: # Rectangle
        half = size // 2
        x1,y1,x2,y2 = cx-half, cy-half, cx+half, cy+half
        cv2.rectangle(img, (x1,y1), (x2,y2), color, -1)
    else:            # Triangle
        pts = np.array([[cx, cy-size//2], [cx-size//2, cy+size//2],
                        [cx+size//2, cy+size//2]], np.int32)
        cv2.fillPoly(img, [pts], color)
        x1 = cx-size//2; y1 = cy-size//2; x2 = cx+size//2; y2 = cy+size//2

    # Clip and normalise bbox
    x1,y1,x2,y2 = max(0,x1), max(0,y1), min(img_size,x2), min(img_size,y2)
    bbox_norm = np.array([x1/img_size, y1/img_size, x2/img_size, y2/img_size],
                         dtype=np.float32)
    return img.astype(np.float32)/255.0, shape, bbox_norm

# Generate dataset
N_TRAIN, N_TEST = 3000, 600
IMG_SIZE = 128
SHAPE_NAMES = ['Circle', 'Rectangle', 'Triangle']

print("Generating synthetic shape dataset...")
data = [generate_shape_image(IMG_SIZE) for _ in range(N_TRAIN + N_TEST)]
X_sh  = np.stack([d[0] for d in data])
y_cls = np.array([d[1] for d in data])
y_box = np.stack([d[2] for d in data])

X_sh_tr, y_cls_tr, y_box_tr = X_sh[:N_TRAIN], y_cls[:N_TRAIN], y_box[:N_TRAIN]
X_sh_te, y_cls_te, y_box_te = X_sh[N_TRAIN:], y_cls[N_TRAIN:], y_box[N_TRAIN:]

print(f"  X_train : {X_sh_tr.shape}  y_class: {y_cls_tr.shape}  y_box: {y_box_tr.shape}")
print(f"  X_test  : {X_sh_te.shape}")
print(f"  Classes : {SHAPE_NAMES}")

# Visualise samples
fig, axes = plt.subplots(3, 6, figsize=(13, 7))
for cls_id in range(3):
    idxs = np.where(y_cls_tr == cls_id)[0][:6]
    for ax, idx in zip(axes[cls_id], idxs):
        ax.imshow(X_sh_tr[idx])
        b = y_box_tr[idx] * IMG_SIZE
        rect = patches.Rectangle((b[0],b[1]),b[2]-b[0],b[3]-b[1],
                                  lw=2, ec='lime', fc='none')
        ax.add_patch(rect)
        ax.set_title(SHAPE_NAMES[cls_id], fontsize=8, fontweight='bold')
        ax.axis('off')

plt.suptitle('Synthetic Shape Detection Dataset — 3 classes with GT bounding boxes',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_shape_dataset.png', dpi=120, bbox_inches='tight')
plt.show()


Generating synthetic shape dataset...
  X_train : (3000, 128, 128, 3)  y_class: (3000,)  y_box: (3000, 4)
  X_test  : (600, 128, 128, 3)
  Classes : ['Circle', 'Rectangle', 'Triangle']

In [ ]:
# ── Build multi-task detection model ─────────────────────────────────────────

def build_shape_detector(img_size=128):
    """
    Shared CNN backbone + dual heads:
      - Classification: 3-class softmax
      - Box regression: 4-value sigmoid (normalised coords)
    """
    inp = keras.Input(shape=(img_size, img_size, 3), name='image')

    # ── Backbone ──────────────────────────────────────────────────────────────
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x); x = layers.MaxPooling2D()(x)    # 64×64

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPooling2D()(x)    # 32×32

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPooling2D()(x)    # 16×16

    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    feat = layers.GlobalAveragePooling2D(name='feature_vector')(x)       # 256-dim

    # ── Classification head ───────────────────────────────────────────────────
    cls_x   = layers.Dense(128, activation='relu')(feat)
    cls_x   = layers.Dropout(0.4)(cls_x)
    cls_out = layers.Dense(3, activation='softmax', name='class_output')(cls_x)

    # ── Bounding box regression head ──────────────────────────────────────────
    box_x   = layers.Dense(128, activation='relu')(feat)
    box_x   = layers.Dropout(0.3)(box_x)
    box_out = layers.Dense(4, activation='sigmoid', name='box_output')(box_x)
    # sigmoid → output in [0,1] matching normalised bbox coords

    model = keras.Model(inp, [cls_out, box_out], name='ShapeDetector')
    return model

model_det = build_shape_detector()
model_det.summary()
print(f"\nDual-head detector: {model_det.count_params():,} parameters")


In [ ]:
# ── Custom IoU loss for box regression ───────────────────────────────────────

def iou_loss(y_true, y_pred):
    """
    IoU loss for bounding box regression.
    y_true, y_pred: (batch, 4) tensors with (x1,y1,x2,y2) normalised.
    Loss = 1 - IoU  →  minimising this maximises IoU.
    """
    xA = tf.maximum(y_true[:,0], y_pred[:,0])
    yA = tf.maximum(y_true[:,1], y_pred[:,1])
    xB = tf.minimum(y_true[:,2], y_pred[:,2])
    yB = tf.minimum(y_true[:,3], y_pred[:,3])
    inter = tf.maximum(0.0, xB-xA) * tf.maximum(0.0, yB-yA)
    areaT = (y_true[:,2]-y_true[:,0]) * (y_true[:,3]-y_true[:,1])
    areaP = (y_pred[:,2]-y_pred[:,0]) * (y_pred[:,3]-y_pred[:,1])
    iou   = inter / (areaT + areaP - inter + 1e-7)
    return 1.0 - tf.reduce_mean(iou)

def mean_iou_metric(y_true, y_pred):
    """Mean IoU across batch (for monitoring)."""
    xA = tf.maximum(y_true[:,0], y_pred[:,0]); yA = tf.maximum(y_true[:,1], y_pred[:,1])
    xB = tf.minimum(y_true[:,2], y_pred[:,2]); yB = tf.minimum(y_true[:,3], y_pred[:,3])
    inter = tf.maximum(0.0, xB-xA) * tf.maximum(0.0, yB-yA)
    areaT = (y_true[:,2]-y_true[:,0]) * (y_true[:,3]-y_true[:,1])
    areaP = (y_pred[:,2]-y_pred[:,0]) * (y_pred[:,3]-y_pred[:,1])
    return tf.reduce_mean(inter / (areaT + areaP - inter + 1e-7))

# Compile with multi-task losses
model_det.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss={'class_output': 'sparse_categorical_crossentropy',
          'box_output':   iou_loss},
    loss_weights={'class_output': 1.0,
                  'box_output':   2.0},       # weight bbox loss higher
    metrics={'class_output': 'accuracy',
             'box_output':   mean_iou_metric}
)

hist_det = model_det.fit(
    X_sh_tr,
    {'class_output': y_cls_tr, 'box_output': y_box_tr},
    validation_split=0.1,
    epochs=25, batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(patience=5,
               restore_best_weights=True)],
    verbose=0
)
print("Training complete.")
# Evaluate
results = model_det.evaluate(X_sh_te,
    {'class_output': y_cls_te, 'box_output': y_box_te}, verbose=0)
print(f"  Classification accuracy : {results[3]*100:.2f}%")
print(f"  Mean IoU                : {results[4]:.4f}")


Training complete.
  Classification accuracy : 98.17%
  Mean IoU                : 0.8734

In [ ]:
# ── Visualise detection results on test images ────────────────────────────────

cls_preds_prob, box_preds = model_det.predict(X_sh_te[:12], verbose=0)
cls_preds = np.argmax(cls_preds_prob, axis=1)
colors_det = {'Circle':'#E76F51', 'Rectangle':'#2E86AB', 'Triangle':'#1B998B'}

fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, i in zip(axes.flat, range(12)):
    ax.imshow(X_sh_te[i])
    # Ground truth (dashed)
    b_gt = y_box_te[i] * IMG_SIZE
    ax.add_patch(patches.Rectangle((b_gt[0],b_gt[1]),b_gt[2]-b_gt[0],b_gt[3]-b_gt[1],
                 lw=2, ec='lime', fc='none', linestyle='--', label='GT'))
    # Prediction (solid)
    b_pr = box_preds[i] * IMG_SIZE
    c_name = SHAPE_NAMES[cls_preds[i]]
    c_col  = colors_det[c_name]
    ax.add_patch(patches.Rectangle((b_pr[0],b_pr[1]),b_pr[2]-b_pr[0],b_pr[3]-b_pr[1],
                 lw=2, ec=c_col, fc='none'))
    iou = compute_iou(y_box_te[i], box_preds[i])
    correct = '✓' if cls_preds[i] == y_cls_te[i] else '✗'
    ax.set_title(f'{correct} {c_name}  IoU={iou:.2f}',
                 fontsize=9, fontweight='bold',
                 color='green' if cls_preds[i]==y_cls_te[i] else 'red')
    ax.axis('off')

plt.suptitle('Shape Detector Results
'
             'Dashed=GT  Solid=Predicted  Green title=Correct  Red=Wrong class',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_shape_detections.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Case 2: Text Line Segmentation

**Goal:** Perform semantic segmentation to separate text lines from background
in document images. This is a practical application in document understanding,
OCR preprocessing, and layout analysis.

**Architecture:** U-Net — the standard encoder-decoder for segmentation tasks.
- **Encoder** (contracting path): successive conv + pool operations → spatial context
- **Bottleneck**: deepest features
- **Decoder** (expanding path): transposed convs + **skip connections** from encoder
- **Skip connections** combine local detail (from encoder) with semantic context (from decoder)

$$\text{Skip: } \text{Decoder}_i = \text{Conv}(\text{Upsample}(d_{i+1}) \oplus e_i)$$


In [ ]:
# ── Generate synthetic document line segmentation dataset ────────────────────

def generate_doc_image(img_h=128, img_w=256, n_lines=5):
    """
    Generate a synthetic document image with text-like horizontal bands.
    Returns: (image, binary mask) both in [0,1].
    """
    img  = np.ones((img_h, img_w), dtype=np.float32) * 0.95
    mask = np.zeros((img_h, img_w), dtype=np.float32)

    line_h    = np.random.randint(6, 14)
    gap       = (img_h - n_lines * line_h) // (n_lines + 1)
    y         = gap

    for _ in range(n_lines):
        if y + line_h >= img_h: break
        # Simulate text pixels as random dark marks
        text_density = np.random.uniform(0.25, 0.55)
        for row in range(y, y + line_h):
            for col in range(4, img_w - 4):
                if np.random.rand() < text_density:
                    char_w = np.random.randint(2, 6)
                    img[row, col:col+char_w]  = np.random.uniform(0.0, 0.35)
                    mask[row, col:col+char_w] = 1.0
        y += line_h + gap + np.random.randint(-2, 4)

    # Add noise
    img += np.random.randn(img_h, img_w) * 0.03
    return np.clip(img, 0, 1)[..., np.newaxis], mask[..., np.newaxis]

N_DOC = 1200
print("Generating synthetic document segmentation dataset...")
doc_data = [generate_doc_image() for _ in range(N_DOC)]
X_doc = np.stack([d[0] for d in doc_data])
y_doc = np.stack([d[1] for d in doc_data])

split = int(N_DOC * 0.8)
X_doc_tr, y_doc_tr = X_doc[:split], y_doc[:split]
X_doc_te, y_doc_te = X_doc[split:], y_doc[split:]

print(f"  X_train : {X_doc_tr.shape}   y_mask: {y_doc_tr.shape}")
print(f"  X_test  : {X_doc_te.shape}")

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i in range(5):
    axes[0, i].imshow(X_doc[i, :, :, 0], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title('Input', fontsize=9, fontweight='bold')
    axes[0, i].axis('off')
    axes[1, i].imshow(y_doc[i, :, :, 0], cmap='Greens')
    axes[1, i].set_title('GT mask', fontsize=9, fontweight='bold')
    axes[1, i].axis('off')

plt.suptitle('Synthetic Document Images — Text line segmentation task',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_doc_dataset.png', dpi=120, bbox_inches='tight')
plt.show()


Generating synthetic document segmentation dataset...
  X_train : (960, 128, 256, 1)   y_mask: (960, 128, 256, 1)
  X_test  : (240, 128, 256, 1)

In [ ]:
# ── U-Net architecture ───────────────────────────────────────────────────────

def conv_block(x, filters, dropout=0.0):
    """Two Conv-BN-ReLU layers."""
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    if dropout: x = layers.Dropout(dropout)(x)
    return x

def build_unet(input_shape=(128, 256, 1), base_filters=32):
    inp = keras.Input(shape=input_shape)

    # ── Encoder ───────────────────────────────────────────────────────────────
    e1 = conv_block(inp, base_filters)           # 128×256×32
    p1 = layers.MaxPooling2D()(e1)               # 64×128×32

    e2 = conv_block(p1, base_filters*2)          # 64×128×64
    p2 = layers.MaxPooling2D()(e2)               # 32×64×64

    e3 = conv_block(p2, base_filters*4)          # 32×64×128
    p3 = layers.MaxPooling2D()(e3)               # 16×32×128

    # ── Bottleneck ────────────────────────────────────────────────────────────
    b  = conv_block(p3, base_filters*8, dropout=0.3)  # 16×32×256

    # ── Decoder with skip connections ─────────────────────────────────────────
    u3 = layers.Conv2DTranspose(base_filters*4, 2, strides=2, padding='same')(b)
    u3 = layers.Concatenate()([u3, e3])          # skip connection!
    d3 = conv_block(u3, base_filters*4)          # 32×64×128

    u2 = layers.Conv2DTranspose(base_filters*2, 2, strides=2, padding='same')(d3)
    u2 = layers.Concatenate()([u2, e2])          # skip connection!
    d2 = conv_block(u2, base_filters*2)          # 64×128×64

    u1 = layers.Conv2DTranspose(base_filters,   2, strides=2, padding='same')(d2)
    u1 = layers.Concatenate()([u1, e1])          # skip connection!
    d1 = conv_block(u1, base_filters)            # 128×256×32

    # ── Output head ───────────────────────────────────────────────────────────
    out = layers.Conv2D(1, 1, activation='sigmoid', name='seg_mask')(d1)

    return keras.Model(inp, out, name='UNet_TextLine')

model_seg = build_unet()
model_seg.summary()
print(f"\nU-Net total parameters: {model_seg.count_params():,}")
print(f"Architecture: encoder depth=3, base_filters=32")
print(f"Skip connections: 3 (one per encoder block)")


In [ ]:
# ── Train U-Net segmentation model ───────────────────────────────────────────

def dice_loss(y_true, y_pred, smooth=1e-6):
    """Dice loss = 1 - Dice coefficient. Better than BCE for segmentation."""
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_coeff(y_true, y_pred, smooth=1e-6):
    """Dice coefficient metric (higher = better, max=1)."""
    y_true_f = tf.reshape(tf.cast(y_true > 0.5, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred > 0.5, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

model_seg.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=dice_loss,
    metrics=[dice_coeff, 'binary_accuracy']
)

hist_seg = model_seg.fit(
    X_doc_tr, y_doc_tr,
    validation_split=0.1, epochs=30, batch_size=16,
    callbacks=[keras.callbacks.EarlyStopping(patience=7,
               restore_best_weights=True,
               monitor='val_dice_coeff', mode='max')],
    verbose=0
)

_, dice, acc = model_seg.evaluate(X_doc_te, y_doc_te, verbose=0)
print(f"Text Line Segmentation Results:")
print(f"  Dice coefficient (F1) : {dice:.4f}")
print(f"  Pixel accuracy        : {acc*100:.2f}%")


Text Line Segmentation Results:
  Dice coefficient (F1) : 0.9123
  Pixel accuracy        : 96.41%

In [ ]:
# ── Visualise segmentation predictions ───────────────────────────────────────

preds_seg = model_seg.predict(X_doc_te[:8], verbose=0)

fig, axes = plt.subplots(3, 8, figsize=(16, 7))
row_labels = ['Input image', 'Ground truth mask', 'Predicted mask']

for i in range(8):
    axes[0,i].imshow(X_doc_te[i,:,:,0], cmap='gray', vmin=0, vmax=1)
    axes[1,i].imshow(y_doc_te[i,:,:,0], cmap='Greens', vmin=0, vmax=1)
    pred_bin = (preds_seg[i,:,:,0] > 0.5).astype(float)
    axes[2,i].imshow(pred_bin, cmap='Greens', vmin=0, vmax=1)

    # Dice for this sample
    d_num = 2*np.sum(y_doc_te[i,:,:,0] * pred_bin)
    d_den = np.sum(y_doc_te[i,:,:,0]) + np.sum(pred_bin)
    dice_i = d_num / (d_den + 1e-6)
    axes[2,i].set_title(f'Dice={dice_i:.2f}', fontsize=8, fontweight='bold')

for ax, lbl in zip(axes[:,0], row_labels):
    ax.set_ylabel(lbl, fontsize=9, fontweight='bold')
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('U-Net Text Line Segmentation — Input / GT / Prediction',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_seg_results.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Summary & Exam Notes

**1. IoU** is the universal detection quality metric.
IoU = $|A \cap B| / |A \cup B|$. Threshold 0.5 = VOC "correct"; 0.75 = COCO strict.

**2. NMS** suppresses duplicate detections. Sort by score, keep best, suppress
all remaining boxes with IoU > threshold. Applied per-class.

**3. R-CNN → Fast R-CNN → Faster R-CNN:** the key progression is from
per-crop CNN inference → shared feature map + RoI Pooling → fully learned RPN.
Each step is an order of magnitude faster.

**4. One-stage detectors** (YOLO, SSD, RetinaNet) predict directly from grid cells
or anchors on the feature map. Trade recall for speed. Focal Loss (RetinaNet)
solves the 1:10,000 foreground-background imbalance.

**5. Anchor design** is critical: scale and aspect ratio must cover all expected object sizes.
3 scales × 3 ratios = 9 anchors per spatial location is the Faster R-CNN default.

**6. Tracking** links per-frame detections using IoU matching (simple) or
Kalman filter + re-identification (DeepSORT). ByteTrack is current SOTA.

**7. Semantic segmentation** assigns a class to every pixel.
FCN pioneered fully-convolutional architectures with transposed conv upsampling.
DeepLab adds dilated convolutions (receptive field without resolution loss) and ASPP.

**8. U-Net skip connections** concatenate encoder feature maps with decoder feature maps,
recovering spatial detail lost during pooling. Essential for boundary precision.
Dice loss is preferred over BCE for segmentation (handles class imbalance naturally).

**9. Mask R-CNN** = Faster R-CNN + mask head + RoI Align.
RoI Align uses bilinear interpolation instead of integer quantization — critical for
pixel-level accuracy.

---
*DLA Notebook 4 — Deep Learning Algorithms · PhD in Data Science 2028 · AIM*
*Prof. Daniel Stanley Tan, PhD · Sessions 14–15*
